In [ ]:
import re
import numpy as np


def load_file_content(filepath):
    try:
        with open(filepath, "r") as f:
            return f.read()
    except FileNotFoundError:
        return None
    
def extract_die_area(def_text):
    for line in def_text.splitlines():
        if line.startswith("DIEAREA"):
            # Extract numbers inside parentheses
            # This finds all sequences of digits
            nums = re.findall(r'\d+', line)
            if len(nums) == 4:
                die_x_min = int(nums[0])
                die_y_min = int(nums[1])
                die_x_max = int(nums[2])
                die_y_max = int(nums[3])
                return die_x_min, die_y_min, die_x_max, die_y_max
    return RuntimeError("DIEAREA not found in DEF file")



In [ ]:
#extra die area 
design_path = "./CTS-Bench/runs/aes_run_20260302_201328//11-openroad-detailedplacement/aes.def"
def_text = load_file_content(design_path)
die_area = extract_die_area(def_text)

Die Area: (0, 0, 739485, 750205)


In [ ]:
import re

def extract_graph_nodes(def_text, clock_port="clk"):
    # 1. Topologically identify all flip-flops via the clock net
    # (Using your exact logic)
    clock_pattern = rf'-\s+{re.escape(clock_port)}\s+\(\s+PIN\s+{re.escape(clock_port)}\s+\).*?;'
    clock_match = re.search(clock_pattern, def_text, re.DOTALL)
    
    if not clock_match:
        print(f"Warning: Clock net '{clock_port}' not found.")
        all_flops = set()
    else:
        # Extracts the instance names connected to the CLK pin
        all_flops = set(re.findall(r'\(\s+((?!PIN)\S+)\s+CLK\s+\)', clock_match.group(0)))
        
    print(f"Topological scan found {len(all_flops)} flip-flops connected to {clock_port}.")

    # 2. Isolate the COMPONENTS block
    block_match = re.search(r'COMPONENTS\s+\d+\s*;(.*?)END COMPONENTS', def_text, re.DOTALL)
    components_block = block_match.group(1) if block_match else ""
    
    # 3. Define Regex Patterns for components
    line_pattern = re.compile(r'^\s*-\s+(\S+)\s+(\S+)(.*?);', re.MULTILINE)
    coord_pattern = re.compile(r'\(\s*(-?\d+)\s+(-?\d+)\s*\)')
    
    extracted_nodes = []
    ignore_keywords = ["decap", "fill", "tap", "tapvpwrvgnd", "diode", "antenna"]

    # 4. Parse components and build features
    for match in line_pattern.finditer(components_block):
        instance_name = match.group(1)
        cell_name = match.group(2)
        properties = match.group(3)
        
        if any(keyword in cell_name.lower() for keyword in ignore_keywords):
            continue
            
        coords_match = coord_pattern.search(properties)
        x = int(coords_match.group(1)) if coords_match else None
        y = int(coords_match.group(2)) if coords_match else None
            
        drive_match = re.search(r'_(\d+)$', cell_name)
        drive_strength = int(drive_match.group(1)) if drive_match else 0
            
        # TOPOLOGICAL CHECK: Is it in the flop set?
        is_sequential = 1 if instance_name in all_flops else 0
        
        is_buffer = 1 if ("buf" in cell_name.lower() or "inv" in cell_name.lower()) else 0
        
        extracted_nodes.append({
            "instance_name": instance_name,
            "cell_name": cell_name,
            "x": x,
            "y": y,
            "drive_strength": drive_strength,
            "is_sequential": is_sequential,
            "is_buffer": is_buffer
        })

    return extracted_nodes, all_flops



extracted_nodes, all_flops = extract_graph_nodes(def_text, clock_port="clk")

# Print the header
print(f"Total Nodes Extracted: {len(extracted_nodes)}")
print("-" * 105)
print(f"{'Instance Name':<20} | {'Cell Name':<30} | {'(X, Y)':<20} | {'Drive':<5} | {'Seq':<3} | {'Buf/Inv':<7}")
print("-" * 105)

# Loop through the list and print each node 
for node in extracted_nodes: 
    coords = f"({node['x']}, {node['y']})"
    print(f"{node['instance_name']:<20} | {node['cell_name']:<30} | {coords:<20} | {node['drive_strength']:<5} | {node['is_sequential']:<3} | {node['is_buffer']:<7}")



Topological scan found 2994 flip-flops connected to clk.
Total Nodes Extracted: 24327
---------------------------------------------------------------------------------------------------------
Instance Name        | Cell Name                      | (X, Y)               | Drive | Seq | Buf/Inv
---------------------------------------------------------------------------------------------------------
_20459_              | sky130_fd_sc_hd__inv_2         | (298540, 269280)     | 2     | 0   | 1      
_20460_              | sky130_fd_sc_hd__inv_2         | (508760, 266560)     | 2     | 0   | 1      
_20461_              | sky130_fd_sc_hd__inv_2         | (317860, 266560)     | 2     | 0   | 1      
_20462_              | sky130_fd_sc_hd__clkinv_16     | (23460, 685440)      | 16    | 0   | 1      
_20463_              | sky130_fd_sc_hd__clkinv_16     | (186300, 552160)     | 16    | 0   | 1      
_20464_              | sky130_fd_sc_hd__inv_6         | (96140, 353600)      | 6     | 0   | 1  